# Case Study: mTOR Approved Molecules

Run the newly trained 80/20 BBB models on the SMILES in `data/mtor_approved.csv`. Rows without SMILES are skipped.

In [1]:
from pathlib import Path
import warnings

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from padelpy import from_smiles

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = ROOT / "data" / "mtor_approved.csv"
MODEL_DIR = ROOT / "output" / "models" / "hypertuning"

models = {
    "KNN": joblib.load(MODEL_DIR / "KNN_8020_best_model.pkl"),
    "LGBM": joblib.load(MODEL_DIR / "LGBM_8020_best_model.pkl"),
    "ET": joblib.load(MODEL_DIR / "ET_8020_best_model.pkl"),
}
feature_names = joblib.load(MODEL_DIR / "feature_names_existing_8020.pkl")

raw = pd.read_csv(DATA_PATH)
smiles_col = "Smiles"
chembl_col = "Parent Molecule ChEMBL ID"

case_df = raw[[chembl_col, smiles_col]].copy()
case_df = case_df.dropna(subset=[smiles_col])
case_df[smiles_col] = case_df[smiles_col].astype(str).str.strip()
case_df = case_df[case_df[smiles_col] != ""].reset_index(drop=True)

print(f"Loaded {len(raw)} rows from {DATA_PATH.name}")
print(f"Running predictions for {len(case_df)} rows with SMILES")


def bbb_label(value):
    return "BBB+" if int(value) == 1 else "BBB-"


def model_prediction(model, descriptor_row):
    pred = int(model.predict(descriptor_row)[0])
    if hasattr(model, "predict_proba"):
        probs = model.predict_proba(descriptor_row)[0]
        confidence = float(probs[pred] * 100)
    else:
        confidence = np.nan
    return pred, confidence


rows = []
for _, row in case_df.iterrows():
    chembl_id = row[chembl_col]
    smiles = row[smiles_col]

    descriptors = pd.DataFrame([from_smiles(smiles, fingerprints=False, descriptors=True)])
    descriptors = descriptors.apply(pd.to_numeric, errors="coerce")
    descriptors = descriptors.replace([np.inf, -np.inf], np.nan).fillna(0)
    descriptors = descriptors.reindex(columns=feature_names, fill_value=0)

    model_results = {}
    for model_name, model in models.items():
        pred, conf = model_prediction(model, descriptors)
        model_results[model_name] = {"prediction": pred, "confidence": conf}

    preds = [result["prediction"] for result in model_results.values()]
    majority_pred = max(set(preds), key=preds.count)
    agreeing_models = sum(pred == majority_pred for pred in preds)
    agreement_pct = agreeing_models / len(preds) * 100
    agreeing_confidences = [
        result["confidence"]
        for result in model_results.values()
        if result["prediction"] == majority_pred and not np.isnan(result["confidence"])
    ]
    all_confidences = [
        result["confidence"]
        for result in model_results.values()
        if not np.isnan(result["confidence"])
    ]

    rows.append({
        "chemblD": chembl_id,
        "smiles": smiles,
        "prediction (BBB+/-)": bbb_label(majority_pred),
        "confidence": round(float(np.mean(agreeing_confidences)), 2) if agreeing_confidences else np.nan,
        "uncertainty": round(float(np.std(all_confidences)), 2) if all_confidences else np.nan,
        "agreement": f"{agreeing_models}/{len(preds)} ({agreement_pct:.1f}%)",
        "KNN prediction": bbb_label(model_results["KNN"]["prediction"]),
        "KNN confidence": round(model_results["KNN"]["confidence"], 2),
        "LGBM prediction": bbb_label(model_results["LGBM"]["prediction"]),
        "LGBM confidence": round(model_results["LGBM"]["confidence"], 2),
        "ET prediction": bbb_label(model_results["ET"]["prediction"]),
        "ET confidence": round(model_results["ET"]["confidence"], 2),
    })

results = pd.DataFrame(rows)
display(results)


Loaded 26 rows from mtor_approved.csv
Running predictions for 22 rows with SMILES


,chemblD,smiles,prediction (BBB+/-),confidence,uncertainty,agreement,KNN prediction,KNN confidence,LGBM prediction,LGBM confidence,ET prediction,ET confidence
0,CHEMBL1258517,O=C(Nc1ccncc1)Nc1ccc(-c2nc(N3CCOCC3)nc(N3C4CCC3COC4)n2)cc1,BBB-,58.99,9.21,2/3 (66.7%),BBB+,78.29,BBB-,60.79,BBB-,57.19
1,CHEMBL5095025,Cn1/c(=N\C#N)n(-c2ccc(C(C)(C)C#N)nc2)c2c3cc(-c4cnc(N)c(C(F)(F)F)c4)ccc3ncc21,BBB-,71.90,16.61,3/3 (100.0%),BBB-,59.99,BBB-,95.38,BBB-,60.32
2,CHEMBL2326966,N=C(N)NCCC[C@H](NC(=O)CCC(=O)OC[N+]1(c2cc(=O)c3cccc(-c4ccccc4)c3o2)CCOCC1)C(=O)NCC(=O)N[C@@H](CC(=O)O)C(=O)N[C@@H](C...,BBB-,90.01,10.69,3/3 (100.0%),BBB-,100.00,BBB-,94.83,BBB-,75.19
3,CHEMBL1236962,COc1ncc(-c2ccc3nccc(-c4ccnnc4)c3c2)cc1NS(=O)(=O)c1ccc(F)cc1F,BBB-,85.62,12.35,3/3 (100.0%),BBB-,100.00,BBB-,87.01,BBB-,69.84
4,CHEMBL2141712,COc1ccc(COc2cc3oc(=O)c4cc(C(C)O)ccc4c3cc2OC)cc1,BBB-,66.14,9.64,3/3 (100.0%),BBB-,60.44,BBB-,79.71,BBB-,58.26
5,CHEMBL2331680,CCNC(=O)Nc1ccc(-c2nc3c(c(N4CCOC[C@@H]4C)n2)CCN(C2COC2)C3)cc1,BBB+,59.25,3.21,2/3 (66.7%),BBB+,60.88,BBB-,53.06,BBB+,57.61
6,CHEMBL592445,CN(C)C1CCN(C(=O)c2ccc(NC(=O)Nc3ccc(-c4nc(N5CCOCC5)nc(N5CCOCC5)n4)cc3)cc2)CC1,BBB-,79.29,12.66,2/3 (66.7%),BBB+,57.10,BBB-,88.03,BBB-,70.56
7,CHEMBL1234354,COc1ccc(-c2cc3c(C)nc(N)nc3n([C@H]3CC[C@H](OCCO)CC3)c2=O)cn1,BBB-,83.12,15.53,3/3 (100.0%),BBB-,100.00,BBB-,86.84,BBB-,62.51
8,CHEMBL1922094,Cc1c(CN2CCN(C(=O)[C@H](C)O)CC2)sc2c(N3CCOCC3)nc(-c3cnc(N)nc3)nc12,BBB-,71.37,12.79,3/3 (100.0%),BBB-,80.34,BBB-,80.49,BBB-,53.28
9,CHEMBL3586573,CCN1C(=O)CNc2ncc(-c3ccc(-c4nc[nH]n4)nc3C)nc21,BBB-,64.46,11.39,3/3 (100.0%),BBB-,59.02,BBB-,80.31,BBB-,54.06
